# 260310 OpenAI API 첫 연결 & 채팅 완성(Chat Completions) 기본기

**오늘의 목표**
- LLM 서비스 12주 과정의 첫날! OpenAI API 키를 환경변수로 안전하게 로드하고, `client.chat.completions.create(...)`로 첫 응답을 받아본다.
- 응답 객체 구조(`ChatCompletion` → `choices` → `message` → `content`)를 한 꺼풀씩 벗겨서 이해한다.
- `role`(system/user/assistant) 개념과 페르소나(system prompt)로 답변 톤을 제어하는 패턴을 익힌다.

**비유 한 줄**: API 호출은 식당에 주문하기와 같다. `client`는 종업원, `messages`는 주문서(누가 어떤 말을 했는지), `model`은 어느 주방(gpt-4o-mini)에 맡길지, `response`는 돌려받은 접시다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260310_openai_api.ipynb)

## 0. 환경 세팅 (Colab / 로컬 공통)

Colab이면 `userdata`로 시크릿에서 API 키를 꺼내 쓰고, 로컬이면 같은 폴더에 `.env` 파일을 만들어 `OPENAI_API_KEY=sk-...` 한 줄을 넣는다.

**왜 `.env`를 쓰나?**: 코드에 키를 그대로 박으면 GitHub에 올라가는 순간 크롤러가 긁어가서 수십~수백만원 요금이 나온 사례가 실제로 많다. `.env`는 `.gitignore`에 넣어 레포에 커밋되지 않게 하고, `os.getenv(...)`로만 꺼내 쓰는 게 업계 표준이다.

**비유 한 줄**: API 키는 신용카드 번호다. 소스코드에 적어두는 건 카페 벽에 카드번호 써 붙이는 것과 같다.

In [ ]:
# Colab 첫 실행 시 필요한 라이브러리 설치 (로컬에서 이미 설치했다면 건너뛰어도 됨)
!pip install -q openai langchain-openai langchain-core python-dotenv

In [ ]:
import os
from openai import OpenAI                          # OpenAI 공식 SDK — API 호출용 클라이언트 클래스
from langchain_openai import ChatOpenAI             # LangChain이 감싼 OpenAI 래퍼 — 이후 LCEL 체인에서 사용
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- API 키 로드 (Colab이면 userdata, 로컬이면 .env) ---
try:
    # Colab 환경: 좌측 열쇠 아이콘에서 'OPENAI_API_KEY' 시크릿을 등록해두면 여기서 읽힘
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception:
    # 로컬 환경: 프로젝트 폴더의 .env 파일을 읽어 환경변수로 주입
    from dotenv import load_dotenv
    load_dotenv()

# --- 클라이언트 및 모델 초기화 ---
client = OpenAI()                                   # 환경변수 OPENAI_API_KEY를 자동으로 집어감
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # temperature=0 → 재현성 높고 결정적인 답변

print("환경 설정 완료")
print(f"모델: gpt-4o-mini")
print(f"API 키 설정: {'OK' if os.environ.get('OPENAI_API_KEY') else 'MISSING'}")

## 1. 첫 번째 호출 — 가장 단순한 user 메시지 하나 보내기

`client.chat.completions.create(...)`는 OpenAI와 통신하는 실제 함수다. 필수 인자는 두 개:
- `model`: 어떤 모델에 보낼지 (우리는 비용이 싸고 빠른 `gpt-4o-mini` 사용)
- `messages`: 대화 히스토리를 **리스트**로 전달. 각 메시지는 `{"role": ..., "content": ...}` **딕셔너리**.

**가격 참고 (강사 설명)**: gpt-4o-mini는 백만 토큰당 입력 $0.15, 출력 $0.60 수준으로 매우 저렴. 비용을 줄이고 싶으면 **출력 토큰을 제한하는 게 제일 효과적** (출력이 입력보다 몇 배 더 비싸기 때문).

In [ ]:
# 가장 간단한 호출: system 없이 user 메시지 하나만 보내보기
response0 = client.chat.completions.create(
    model="gpt-4o-mini",                  # 사용할 모델 지정
    messages=[                            # 대화는 항상 '리스트' 안에 '딕셔너리' 형태
        {
            "role": "user",               # 누가 말했는지 — user는 사용자(나)
            "content": "안녕하세요. 자기소개 부탁드려요~!"  # 실제 발화 내용
        }
    ]
)

In [ ]:
# 응답 객체 전체를 그대로 찍어보기 → ChatCompletion(id=..., choices=[...], usage=...) 구조 확인
response0

## 2. 응답 구조 한 꺼풀씩 벗겨보기

`response`는 `ChatCompletion` 객체 통째로 — 메타데이터까지 다 들어있어 서비스에서 그대로 쓰기엔 너무 장황하다. 우리가 화면에 보여주고 싶은 건 딱 `message.content` 문자열 하나다.

**비유 한 줄**: 응답은 택배 상자다. 겉포장(ChatCompletion) → 완충재(choices 리스트) → 내부박스(message) → 실제 물건(content). 한 겹씩 뜯어내면 된다.

**중요**: `choices`는 **리스트**다. 파이썬 리스트는 0부터 시작하므로 첫 번째 선택지는 `choices[0]`.

In [ ]:
# 1단계: choices 리스트 꺼내기 — 보통 길이 1짜리 리스트
response0.choices

In [ ]:
# 2단계: 첫 번째(0번째) 선택지만 꺼내기
response0.choices[0]

In [ ]:
# 3단계: 그 선택지 안의 message 객체 꺼내기
response0.choices[0].message

In [ ]:
# 4단계: 최종적으로 우리가 원하는 텍스트만 — message.content
response0.choices[0].message.content

### 잠깐 파이썬 리스트 인덱싱 복습
`choices[0]`이 왜 '첫 번째'인지 감이 안 온다면 — 파이썬은 0부터 센다.

In [ ]:
list_a = [1, 2, 3, 4, 5]   # 리스트는 값 여러 개를 순서대로 모아둔 자료구조
list_a[0]                   # 첫 번째 원소 → 1

In [ ]:
list_a[3]   # 0,1,2,3 → 네 번째 원소인 4

## 3. 페르소나(system role) 부여하기

`role="user"`만 쓰면 LLM은 아무 캐릭터 없이 일반적인 답을 준다. `role="system"` 메시지를 앞에 하나 끼워넣으면 "너는 이런 사람처럼 답해"라고 지시할 수 있다. 논문에서도 페르소나를 주면 답변 품질이 올라간다는 결과가 있다.

**비유 한 줄**: 같은 질문이라도 친구에게 묻는 것과 전문가에게 묻는 것은 답이 다르다. system 메시지는 "지금부터 너는 10년차 베테랑 수의사야" 같은 배역 지정이다.

**세 가지 role 요약**
- `system`: 모델에게 주는 지시/페르소나 (사용자에겐 보통 안 보임)
- `user`: 실제 사용자의 질문
- `assistant`: 모델이 이전에 한 답변 (대화 이력을 유지할 때 사용)

In [ ]:
# system 프롬프트로 '파이썬 전문 프롬프트 엔지니어' 페르소나 부여
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",                                 # 배역 지정 — 모델이 응답할 '입장'
            "content": "당신은 LLM 서비스를 가르치는 파이썬 전문 프롬프트 엔지니어야."
        },
        {
            "role": "user",                                    # 내가 실제로 던지는 질문
            "content": "* RAG이 뭔가요? LangChain은 뭔가요?"
        }
    ]
)

In [ ]:
# 응답 텍스트 부분만 꺼내보기 — 이전보다 전문적 톤으로 답변
response1.choices[0].message.content

## 4. 단계별 답변을 유도하는 페르소나 예제

system에 "항상 단계별로 설명해주세요" 같은 답변 형식 지시를 넣으면 출력 구조까지 제어할 수 있다. 이게 프롬프트 엔지니어링의 출발점이다.

In [ ]:
# 초등학생에게 설명하는 친절한 수학 선생님 페르소나 + 단계별 설명 지시
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            # 페르소나 + 답변 형식 힌트를 system에 함께 넣어두면 일관된 톤을 유지
            "content": "당신은 초등학생을 가르치는 친절한 수학선생님 입니다. 항상 단계별로 설명해주세요"
        },
        {
            "role": "user",
            "content": "1+1은?"
        }
    ]
)

# print()로 감싸면 \n(줄바꿈)이 실제 줄바꿈으로 예쁘게 렌더링된다
print(response2.choices[0].message.content)

### \n(개행 문자) 한 줄 팁
문자열 안의 `\n`은 파이썬에서 '엔터'를 의미한다. 그냥 변수로 찍으면 `\n` 그대로 보이지만 `print()`로 찍으면 실제 줄바꿈으로 출력된다.

In [ ]:
# \n 동작 확인 — print를 써야 진짜 줄바꿈이 일어난다
print("나는 학생입니다\n나는 학생이 아닙니다")

## 5. (연습) 내 페르소나로 바꿔보기

아픈 강아지 증상을 물어본다면? 페르소나를 '10년차 베테랑 수의사'로 설정해보자. 강사님이 수업 중 든 예시 그대로 따라해본다.

**비유 한 줄**: 같은 증상이라도 옆집 아저씨에게 물으면 "글쎄" 하고 끝나지만, 수의사 페르소나를 주면 구조적으로 원인·대처·병원 방문 시점까지 정리해준다.

In [ ]:
# 연습: 페르소나를 바꿔가며 같은 질문에 답변이 어떻게 달라지는지 관찰
response3 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "당신은 10년차 베테랑 수의사입니다. 보호자가 이해하기 쉬운 용어로 원인/대처/병원 방문 시점을 순서대로 안내해주세요."
        },
        {
            "role": "user",
            "content": "저희 강아지가 붉은 반점이 있고, 자꾸 발톱이 빠져요. 어떻게 해야 할까요?"
        }
    ]
)

print(response3.choices[0].message.content)

## 6. 오늘 정리

- `client.chat.completions.create(model=..., messages=[...])` — 이게 오늘 내내 썼던 **딱 하나의 핵심 함수**다.
- `messages`는 `[{role, content}, ...]` 형식의 **리스트 of 딕셔너리**.
- 응답에서 실제 답변 텍스트는 `response.choices[0].message.content`.
- `role`에는 `system`(페르소나/지시) / `user`(질문) / `assistant`(이전 답변) 세 종류.
- API 키는 절대 코드에 박지 말고 `.env`나 Colab userdata에서 불러오기.
- **비용 팁**: 출력 토큰이 입력보다 5배 이상 비싸다 → `max_tokens` 등으로 출력을 제한하는 게 비용 절감 1순위.

내일(3/11)부터는 파이썬 기본 문법(리스트/딕셔너리/함수/try-except)을 정리하면서, 위 호출 패턴을 함수로 감싸고 메시지를 동적으로 만드는 연습으로 이어진다.